In [1]:
import sys
sys.executable

'/Users/agastyanath/Downloads/ns_jupy/bin/python3.14'

In [143]:
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from networkx.algorithms import bipartite, community
import os, random
 
#random.seed(42)
#np.random.seed(42)

In [144]:
DATA_PATH1 = "./web-of-life_2026-03-23_105349/M_PL_001.csv"
DATA_PATH2 = "./web-of-life_2026-03-23_105349/M_PL_038.csv"
OUT_DIR   = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
 
df1 = pd.read_csv(DATA_PATH1, index_col=0)
df2 = pd.read_csv(DATA_PATH2, index_col=0)
df1 = df1.fillna(0).astype(float)
df2 = df2.fillna(0).astype(float)
 
plants      = list(df1.index)
pollinators = list(df1.columns)
n_plants    = len(plants)
n_poll      = len(pollinators)
 
print(f"Dataset: {n_plants} plants × {n_poll} pollinators  |  "
      f"{int((df1>0).sum().sum())} interactions")
 

Dataset: 84 plants × 101 pollinators  |  361 interactions


In [145]:
def build_bipartite(matrix_df):
    """Return a weighted bipartite NetworkX graph."""
    G = nx.Graph()
    pl  = list(matrix_df.index)
    pol = list(matrix_df.columns)
    G.add_nodes_from(pl,  bipartite=0, kind="plant")
    G.add_nodes_from(pol, bipartite=1, kind="pollinator")
    for p in pl:
        for q in pol:
            w = matrix_df.loc[p, q]
            if w > 0:
                G.add_edge(p, q, weight=float(w))
    return G
 
G_orig1 = build_bipartite(df1)
G_orig2 = build_bipartite(df2)
print(f"Graph: {G_orig2.number_of_nodes()} nodes, {G_orig2.number_of_edges()} edges")

Graph: 50 nodes, 79 edges


In [146]:
list(G_orig1.nodes(data=True))

[('Acaena pinn', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia aconcaguensis', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia brachy', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia conferta', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia exilis', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia montana', {'bipartite': 0, 'kind': 'plant'}),
 ('Adesmia radicifolia', {'bipartite': 0, 'kind': 'plant'}),
 ('Alstroemeria pallida', {'bipartite': 0, 'kind': 'plant'}),
 ('Anarthrophyllum cumingii', {'bipartite': 0, 'kind': 'plant'}),
 ('Asteriscium aemocarpon', {'bipartite': 0, 'kind': 'plant'}),
 ('Astragalus curvicaulis', {'bipartite': 0, 'kind': 'plant'}),
 ('Astragalus sp1 M_PL_001', {'bipartite': 0, 'kind': 'plant'}),
 ('Barneoudia chilensis', {'bipartite': 0, 'kind': 'plant'}),
 ('Berberis empetrifolia', {'bipartite': 0, 'kind': 'plant'}),
 ('Berberis montana', {'bipartite': 0, 'kind': 'plant'}),
 ('Bowlesia tropaeolifolia', {'bipartite': 0, 'kind': 'plant'}),
 ('Calandrinia affinis', {'

In [147]:
import pandas as pd

def graph_to_matrix(G):
    # Identify node sets
    plants = [n for n, d in G.nodes(data=True) if d.get('bipartite') == 0]
    polls  = [n for n, d in G.nodes(data=True) if d.get('bipartite') == 1]

    # Initialize empty matrix
    mat = pd.DataFrame(0, index=plants, columns=polls)

    # Fill interactions
    for u, v in G.edges():
        if u in plants and v in polls:
            mat.loc[u, v] = 1
        elif v in plants and u in polls:
            mat.loc[v, u] = 1

    return mat

def cascading_extinction(G, removal_order="degree"):
    """
    Simulate sequential pollinator removal directly on graph.
    
    A plant goes extinct when it has no remaining pollinators.
    
    Returns:
        frac_removed, frac_surviving, n_components
    """

    # Copy graph so original doesn't get nuked
    G_sim = G.copy()

    # Identify node sets
    plants = [n for n, d in G_sim.nodes(data=True) if d.get('bipartite') == 0]
    polls  = [n for n, d in G_sim.nodes(data=True) if d.get('bipartite') == 1]

    n_poll_total  = len(polls)
    n_plant_total = len(plants)

    # Determine removal order
    if removal_order == "degree":
        order = sorted(polls, key=lambda x: G_sim.degree(x), reverse=True)
    elif removal_order == "random":
        order = polls.copy()
        random.shuffle(order)
    else:
        order = polls.copy()

    # Tracking
    frac_removed   = [0.0]
    frac_surviving = [1.0]
    n_components   = [nx.number_connected_components(G_sim)]

    for i, poll in enumerate(order):

        if poll not in G_sim:
            continue

        # Remove pollinator
        G_sim.remove_node(poll)

        # Secondary extinction: plants with no neighbors
        dead_plants = [
            p for p in plants
            if p in G_sim and G_sim.degree(p) == 0
        ]

        G_sim.remove_nodes_from(dead_plants)

        # Update tracking
        frac_removed.append((i + 1) / n_poll_total)

        surviving_plants = [
            n for n, d in G_sim.nodes(data=True)
            if d.get('bipartite') == 0
        ]

        frac_surviving.append(len(surviving_plants) / n_plant_total)

        nc = nx.number_connected_components(G_sim) if G_sim.number_of_nodes() > 0 else 0
        n_components.append(nc)

    return frac_removed, frac_surviving, n_components
    
print("\nRunning cascading extinction simulations…")
fr_base, fs_base, nc_base = cascading_extinction(G_orig2, "degree")


Running cascading extinction simulations…


In [148]:
id='Caenohalictus sp1 M_PL_001'

In [149]:
G_orig1.remove_node(id)
G_new = G_orig2.copy()

In [150]:
G_new.add_node(id, bipartite=1, kind="pollinator")

In [151]:
import random

def assign_random_interactions(G, species, plant_nodes, k=5):
    """
    Randomly connect species to k plants.
    """
    k = min(k, len(plant_nodes))
    chosen = random.sample(plant_nodes, k)

    for plant in chosen:
        G.add_edge(species, plant)

    return G

def assign_random_degree_based(G, species, plant_nodes, avg_degree):
    """
    Assign number of interactions based on network average.
    """
    k = max(1, int(np.random.poisson(avg_degree)))
    k = min(k, len(plant_nodes))

    chosen = random.sample(plant_nodes, k)

    for plant in chosen:
        G.add_edge(species, plant)

    return G

def assign_preferential(G, species, plant_nodes, k=5):
    """
    Preferential attachment: high-degree plants more likely.
    """
    degrees = np.array([G.degree(p) for p in plant_nodes], dtype=float)
    probs = degrees / degrees.sum()

    chosen = np.random.choice(plant_nodes, size=min(k, len(plant_nodes)), replace=False, p=probs)

    for plant in chosen:
        G.add_edge(species, plant)

    return G

plants = [n for n, d in G_new.nodes(data=True) if d.get('bipartite') == 0]
#G_orig2 = assign_random_interactions(G_orig2, id, plants, k=5)

avg_deg = np.mean([d for n, d in G_new.degree() if G_new.nodes[n].get('bipartite') == 1])
#G_orig2 = assign_preferential(G_orig2, id, plants, avg_deg)

G_new = assign_preferential(G_new, id, plants, k=15)

In [152]:
degree = G_new.degree(id)
survives = degree > 0

In [153]:
fr_new, fs_new, nc_new = cascading_extinction(G_new, "degree")

In [154]:
print(degree)

8


In [155]:
print("Degree:", G_new.degree(id))
print("Neighbors:", list(G_new.neighbors(id)))

Degree: 8
Neighbors: [np.str_('Stachys sylvatica'), np.str_('Chaerophyllum temulentum'), np.str_('Angelica sylvestris'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Vicia sepium'), np.str_('Geranium robertianum'), np.str_('Veronica agrestis')]


In [156]:
print(graph_to_matrix(G_new).shape)
print(graph_to_matrix(G_new).sum().sum())

(8, 43)
87


In [157]:
impact = fs_new[-1] - fs_base[-1]
print("Change in plant survival:", impact)

Change in plant survival: 0.0


In [158]:
print("Components change:", nc_new[-1] - nc_base[-1])

Components change: 0


In [159]:
import matplotlib.pyplot as plt

plt.plot(fr_base, fs_base, label="baseline")
plt.plot(fr_new, fs_new, label="with relocation")
plt.legend()
plt.show()

In [160]:
print(fr_base)

[0.0, 0.023809523809523808, 0.047619047619047616, 0.07142857142857142, 0.09523809523809523, 0.11904761904761904, 0.14285714285714285, 0.16666666666666666, 0.19047619047619047, 0.21428571428571427, 0.23809523809523808, 0.2619047619047619, 0.2857142857142857, 0.30952380952380953, 0.3333333333333333, 0.35714285714285715, 0.38095238095238093, 0.40476190476190477, 0.42857142857142855, 0.4523809523809524, 0.47619047619047616, 0.5, 0.5238095238095238, 0.5476190476190477, 0.5714285714285714, 0.5952380952380952, 0.6190476190476191, 0.6428571428571429, 0.6666666666666666, 0.6904761904761905, 0.7142857142857143, 0.7380952380952381, 0.7619047619047619, 0.7857142857142857, 0.8095238095238095, 0.8333333333333334, 0.8571428571428571, 0.8809523809523809, 0.9047619047619048, 0.9285714285714286, 0.9523809523809523, 0.9761904761904762, 1.0]


In [161]:
print(fr_new)

[0.0, 0.023255813953488372, 0.046511627906976744, 0.06976744186046512, 0.09302325581395349, 0.11627906976744186, 0.13953488372093023, 0.16279069767441862, 0.18604651162790697, 0.20930232558139536, 0.23255813953488372, 0.2558139534883721, 0.27906976744186046, 0.3023255813953488, 0.32558139534883723, 0.3488372093023256, 0.37209302325581395, 0.3953488372093023, 0.4186046511627907, 0.4418604651162791, 0.46511627906976744, 0.4883720930232558, 0.5116279069767442, 0.5348837209302325, 0.5581395348837209, 0.5813953488372093, 0.6046511627906976, 0.627906976744186, 0.6511627906976745, 0.6744186046511628, 0.6976744186046512, 0.7209302325581395, 0.7441860465116279, 0.7674418604651163, 0.7906976744186046, 0.813953488372093, 0.8372093023255814, 0.8604651162790697, 0.8837209302325582, 0.9069767441860465, 0.9302325581395349, 0.9534883720930233, 0.9767441860465116, 1.0]


In [162]:
import numpy as np
min_len = min(len(fs_base), len(fs_new))

impact_curve = np.array(fs_new[:min_len]) - np.array(fs_base[:min_len])
print("Max difference:", impact_curve.max())

Max difference: 0.25


In [163]:
from scipy.interpolate import interp1d

f_base = interp1d(fr_base, fs_base, kind='linear', fill_value="extrapolate")
fs_base_interp = f_base(fr_new)

impact_curve = np.array(fs_new) - fs_base_interp
print("Max difference:", impact_curve.max())

Max difference: 0.12790697674418594


In [164]:
print(len(fr_base), len(fr_new))
print(fr_base[-1], fr_new[-1])

43 44
1.0 1.0


In [142]:
results = []
seen = set()

for _ in range(50):
    # compute degrees of plants
    degrees = np.array([G_orig2.degree(p) for p in plants], dtype=float)
    probs = degrees / degrees.sum()

    # sample k plants (same as your assign_preferential)
    chosen = np.random.choice(plants, size=min(5, len(plants)), replace=False, p=probs)

    # store as sorted tuple so order doesn't matter
    chosen_tuple = tuple(sorted(chosen))
    seen.add(chosen_tuple)

print("Unique interaction sets:", len(seen))

for _ in range(50):
    print(sorted(G_temp.neighbors(id)))
    G_temp = G_orig2.copy()
    G_temp.add_node(id, bipartite=1, kind="pollinator")
    G_temp = assign_preferential(G_temp, id, plants, k=5)

    fr_new, fs_new, _ = cascading_extinction(G_temp)

    # interpolate baseline
    from scipy.interpolate import interp1d
    f_base = interp1d(fr_base, fs_base, fill_value="extrapolate")
    fs_base_interp = f_base(fr_new)

    impact_curve = np.array(fs_new) - fs_base_interp
    results.append(impact_curve.max())

print("Mean impact:", np.mean(results))
print("Std:", np.std(results))

Unique interaction sets: 18
[np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Veronica agrestis')]
[np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Geranium robertianum'), np.str_('Silene dioica'), np.str_('Vicia sepium')]
[np.str_('Geranium robertianum'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Stachys sylvatica'), np.str_('Vicia sepium')]
[np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Geranium robertianum'), np.str_('Lapsana communis'), np.str_('Veronica agrestis')]
[np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Silene dioica'), np.str_('Veronica agrestis'), np.str_('Vicia sepium')]
[np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Stachys sylvatica')]
[np.str_('Angelica sylvestris'), np.str_('Chaerophy

In [165]:
results = []
seen = set()

for _ in range(50):
    G_temp = G_orig2.copy()
    G_temp.add_node(id, bipartite=1, kind="pollinator")
    G_temp = assign_preferential(G_temp, id, plants, k=5)

    chosen = tuple(sorted(G_temp.neighbors(id)))
    print(chosen)  # now this is correct
    seen.add(chosen)

    fr_new, fs_new, _ = cascading_extinction(G_temp)

    from scipy.interpolate import interp1d
    f_base = interp1d(fr_base, fs_base, fill_value="extrapolate")
    fs_base_interp = f_base(fr_new)

    impact_curve = np.array(fs_new) - fs_base_interp
    results.append(impact_curve.max())

print("Unique interaction sets:", len(seen))
print("Mean impact:", np.mean(results))
print("Std:", np.std(results))

(np.str_('Angelica sylvestris'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Stachys sylvatica'), np.str_('Veronica agrestis'))
(np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Lapsana communis'), np.str_('Silene dioica'), np.str_('Stachys sylvatica'))
(np.str_('Angelica sylvestris'), np.str_('Geranium robertianum'), np.str_('Lapsana communis'), np.str_('Veronica agrestis'), np.str_('Vicia sepium'))
(np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Geranium robertianum'), np.str_('Lapsana communis'), np.str_('Silene dioica'))
(np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Silene dioica'), np.str_('Stachys sylvatica'), np.str_('Veronica agrestis'))
(np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('Geranium robertianum'), np.str_('Silene dioica'), np.str_('Vicia sepium'))
(np.str_('Angelica sylvestris'), np.str_('Chaerophyllum temulentum'), np.str_('